## Outlier Detection and Data Quality

### Import Libraries

In [31]:
from pathlib import Path

import numpy as np
import pandas as pd

### Define File Paths

In [32]:


PROJECT_DIR = Path.cwd()

INPUT_DIR = (
    PROJECT_DIR
    / "cleaned_data"
    / "feature_engineering"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "cleaned_data"
    / "outlier_detection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Sold dataset
SOLD_INPUT_FILE = (
    INPUT_DIR
    / "sold_residential_features_enriched.csv"
)

# List dataset
LIST_INPUT_FILE = (
    INPUT_DIR
    / "list_residential_features_enriched.csv"
)

# Confirm dataset is exist
print("Sold input file exists:", SOLD_INPUT_FILE.exists())
print("List input file exists:", LIST_INPUT_FILE.exists())

Sold input file exists: True
List input file exists: True


### Load Sold & List Dataset

In [33]:
sold_df = pd.read_csv(
    SOLD_INPUT_FILE,
    low_memory=False
)

# Avoid naming the List dataset "list" because "list" is a built-in Python data type.
# Use list_df instead
list_df = pd.read_csv(
    LIST_INPUT_FILE,
    low_memory=False
)

print("Sold shape:", sold_df.shape)
print("List shape:", list_df.shape)

Sold shape: (447769, 99)
List shape: (615316, 83)


In [34]:
# Create copies of the Sold and List datasets so the original loaded datasets remain unchanged.
sold_flagged = sold_df.copy()
list_flagged = list_df.copy()

sold_iqr_columns = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

list_iqr_columns = [
    "ListPrice",
    "LivingArea",
    "DaysOnMarket"
]

print("Sold working shape:", sold_flagged.shape)
print("List working shape:", list_flagged.shape)

Sold working shape: (447769, 99)
List working shape: (615316, 83)


### Define the IQR Outlier Detection Function

Calculate the IQR boundaries and create a separate outlier flag for each numeric field.


In [35]:
def add_iqr_outlier_flag(
    df,
    column,
    flag_column,
    valid_mask,
    dataset_name,
    multiplier=1.5
):
    """
    Calculate IQR boundaries and add an outlier flag.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset receiving the outlier flag.
    column : str
        Numeric field used for IQR detection.
    flag_column : str
        Name of the new outlier flag.
    valid_mask : pandas.Series
        Condition identifying valid values.
    dataset_name : str
        Dataset label used in the summary.
    multiplier : float
        IQR multiplier. The standard value is 1.5.

    Returns
    -------
    dict
        Summary of the calculated IQR thresholds.
    """

    valid_values = df.loc[
        valid_mask
        & df[column].notna(),
        column
    ]

    if valid_values.empty:
        raise ValueError(
            f"No valid values are available for {column}."
        )

    q1 = valid_values.quantile(0.25)
    median = valid_values.quantile(0.50)
    q3 = valid_values.quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - multiplier * iqr
    upper_bound = q3 + multiplier * iqr

    df[flag_column] = (
        valid_mask
        & df[column].notna()
        & (
            df[column].lt(lower_bound)
            | df[column].gt(upper_bound)
        )
    )

    outlier_count = int(
        df[flag_column].sum()
    )

    valid_count = int(
        valid_values.shape[0]
    )

    return {
        "dataset": dataset_name,
        "column": column,
        "valid_count": valid_count,
        "q1": q1,
        "median": median,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": outlier_count,
        "outlier_percentage": (
            outlier_count
            / valid_count
            * 100
        )
    }

### Apply IQR Detection to the Sold Dataset

Apply IQR detection to ClosePrice, LivingArea, and DaysOnMarket using valid business-rule values.

In [36]:
sold_iqr_results = []

sold_iqr_results.append(
    add_iqr_outlier_flag(
        df=sold_flagged,
        column="ClosePrice",
        flag_column="close_price_iqr_outlier_flag",
        valid_mask=sold_flagged["ClosePrice"].gt(0),
        dataset_name="Sold"
    )
)

sold_iqr_results.append(
    add_iqr_outlier_flag(
        df=sold_flagged,
        column="LivingArea",
        flag_column="living_area_iqr_outlier_flag",
        valid_mask=sold_flagged["LivingArea"].gt(0),
        dataset_name="Sold"
    )
)

sold_iqr_results.append(
    add_iqr_outlier_flag(
        df=sold_flagged,
        column="DaysOnMarket",
        flag_column="days_on_market_iqr_outlier_flag",
        valid_mask=sold_flagged["DaysOnMarket"].ge(0),
        dataset_name="Sold"
    )
)

sold_iqr_summary = pd.DataFrame(
    sold_iqr_results
)

display(sold_iqr_summary)

,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,Sold,ClosePrice,447767,575000.0,825000.0,1300000.0,725000.0,-512500.0,2387500.0,33477,7.476433
1,Sold,LivingArea,447516,1248.0,1646.0,2224.0,976.0,-216.0,3688.0,19568,4.372581
2,Sold,DaysOnMarket,447769,8.0,18.0,48.0,40.0,-52.0,108.0,34144,7.625360


### Apply IQR Detection to the List Dataset

Apply IQR detection to ListPrice, LivingArea, and DaysOnMarket using valid business-rule values.

In [37]:
list_iqr_results = []

list_iqr_results.append(
    add_iqr_outlier_flag(
        df=list_flagged,
        column="ListPrice",
        flag_column="list_price_iqr_outlier_flag",
        valid_mask=list_flagged["ListPrice"].gt(0),
        dataset_name="List"
    )
)

list_iqr_results.append(
    add_iqr_outlier_flag(
        df=list_flagged,
        column="LivingArea",
        flag_column="living_area_iqr_outlier_flag",
        valid_mask=list_flagged["LivingArea"].gt(0),
        dataset_name="List"
    )
)

list_iqr_results.append(
    add_iqr_outlier_flag(
        df=list_flagged,
        column="DaysOnMarket",
        flag_column="days_on_market_iqr_outlier_flag",
        valid_mask=list_flagged["DaysOnMarket"].ge(0),
        dataset_name="List"
    )
)

list_iqr_summary = pd.DataFrame(
    list_iqr_results
)

display(list_iqr_summary)

,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,List,ListPrice,615316,581000.0,849000.0,1385000.0,804000.0,-625000.0,2591000.0,51489,8.367896
1,List,LivingArea,614693,1248.0,1673.0,2304.0,1056.0,-336.0,3888.0,30305,4.930103
2,List,DaysOnMarket,615316,5.0,11.0,22.0,17.0,-20.5,47.5,47782,7.765441


In [38]:
# Create Overall Outlier Flags
# Combine the individual IQR flags to identify records containing at least one statistical outlier.

sold_iqr_flag_columns = [
    "close_price_iqr_outlier_flag",
    "living_area_iqr_outlier_flag",
    "days_on_market_iqr_outlier_flag"
]

list_iqr_flag_columns = [
    "list_price_iqr_outlier_flag",
    "living_area_iqr_outlier_flag",
    "days_on_market_iqr_outlier_flag"
]

# Create an overall Sold outlier flag.
# The flag is True when at least one of the selected
# Sold IQR outlier flags is True for that record.
sold_flagged["any_iqr_outlier_flag"] = (
    sold_flagged[sold_iqr_flag_columns]
    .any(axis=1)
)

# Create an overall List outlier flag.
# The flag is True when at least one of the selected
# List IQR outlier flags is True for that record.
list_flagged["any_iqr_outlier_flag"] = (
    list_flagged[list_iqr_flag_columns]
    .any(axis=1)
)

### Validate Outlier Detection Results

In [39]:
print("Sold IQR outlier counts:")

display(
    sold_flagged[
        sold_iqr_flag_columns
        + ["any_iqr_outlier_flag"]
    ]
    .sum()
    .to_frame("count")
)

print("List IQR outlier counts:")

display(
    list_flagged[
        list_iqr_flag_columns
        + ["any_iqr_outlier_flag"]
    ]
    .sum()
    .to_frame("count")
)

Sold IQR outlier counts:


,count
close_price_iqr_outlier_flag,33477
living_area_iqr_outlier_flag,19568
days_on_market_iqr_outlier_flag,34144
any_iqr_outlier_flag,70296


List IQR outlier counts:


,count
list_price_iqr_outlier_flag,51489
living_area_iqr_outlier_flag,30305
days_on_market_iqr_outlier_flag,47782
any_iqr_outlier_flag,102567


In [40]:
display(sold_iqr_summary)
display(list_iqr_summary)

,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,Sold,ClosePrice,447767,575000.0,825000.0,1300000.0,725000.0,-512500.0,2387500.0,33477,7.476433
1,Sold,LivingArea,447516,1248.0,1646.0,2224.0,976.0,-216.0,3688.0,19568,4.372581
2,Sold,DaysOnMarket,447769,8.0,18.0,48.0,40.0,-52.0,108.0,34144,7.625360


,dataset,column,valid_count,q1,median,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_percentage
0,List,ListPrice,615316,581000.0,849000.0,1385000.0,804000.0,-625000.0,2591000.0,51489,8.367896
1,List,LivingArea,614693,1248.0,1673.0,2304.0,1056.0,-336.0,3888.0,30305,4.930103
2,List,DaysOnMarket,615316,5.0,11.0,22.0,17.0,-20.5,47.5,47782,7.765441


### Create Full Flagged and Filtered Datasets

Preserve all records in the full flagged datasets and create separate filtered datasets by excluding records with at least one IQR outlier.

In [41]:
# Create full flagged datasets that preserve all records and IQR outlier flags
sold_full_flagged = sold_flagged.copy()
list_full_flagged = list_flagged.copy()

# Create filtered Sold dataset by removing records with any IQR outlier
sold_filtered = sold_full_flagged.loc[
    ~sold_full_flagged["any_iqr_outlier_flag"]
].copy()

# Create filtered List dataset by removing records with any IQR outlier
list_filtered = list_full_flagged.loc[
    ~list_full_flagged["any_iqr_outlier_flag"]
].copy()

print("Sold full flagged shape:", sold_full_flagged.shape)
print("Sold filtered shape:", sold_filtered.shape)

print("List full flagged shape:", list_full_flagged.shape)
print("List filtered shape:", list_filtered.shape)

Sold full flagged shape: (447769, 103)
Sold filtered shape: (377473, 103)
List full flagged shape: (615316, 87)
List filtered shape: (512749, 87)


### Compare Row Counts Before and After Filtering

Compare dataset sizes before and after IQR filtering to measure how many records were removed.

In [42]:
row_count_comparison = pd.DataFrame([
    {
        "dataset": "Sold",
        "rows_before": len(sold_full_flagged),
        "rows_after": len(sold_filtered),
        "rows_removed": (
            len(sold_full_flagged)
            - len(sold_filtered)
        ),
        "removed_percentage": (
            (
                len(sold_full_flagged)
                - len(sold_filtered)
            )
            / len(sold_full_flagged)
            * 100
        )
    },
    {
        "dataset": "List",
        "rows_before": len(list_full_flagged),
        "rows_after": len(list_filtered),
        "rows_removed": (
            len(list_full_flagged)
            - len(list_filtered)
        ),
        "removed_percentage": (
            (
                len(list_full_flagged)
                - len(list_filtered)
            )
            / len(list_full_flagged)
            * 100
        )
    }
])

display(row_count_comparison)

,dataset,rows_before,rows_after,rows_removed,removed_percentage
0,Sold,447769,377473,70296,15.699166
1,List,615316,512749,102567,16.668996


### Compare Median Values Before and After Filtering

Compare median values of the selected numeric fields before and after IQR filtering to evaluate the impact of outlier removal.

In [43]:
median_comparison = []

# Compare Sold median values
for column in sold_iqr_columns:
    median_before = sold_full_flagged[column].median()
    median_after = sold_filtered[column].median()

    median_comparison.append({
        "dataset": "Sold",
        "column": column,
        "median_before": median_before,
        "median_after": median_after,
        "median_difference": (
            median_after - median_before
        )
    })

# Compare List median values
for column in list_iqr_columns:
    median_before = list_full_flagged[column].median()
    median_after = list_filtered[column].median()

    median_comparison.append({
        "dataset": "List",
        "column": column,
        "median_before": median_before,
        "median_after": median_after,
        "median_difference": (
            median_after - median_before
        )
    })

median_comparison = pd.DataFrame(
    median_comparison
)

display(median_comparison)

,dataset,column,median_before,median_after,median_difference
0,Sold,ClosePrice,825000.0,787500.0,-37500.0
1,Sold,LivingArea,1646.0,1572.0,-74.0
2,Sold,DaysOnMarket,18.0,16.0,-2.0
3,List,ListPrice,849000.0,798000.0,-51000.0
4,List,LivingArea,1673.0,1587.0,-86.0
5,List,DaysOnMarket,11.0,9.0,-2.0


### Check for Close to Original List Price Ratio Data Quality

In [70]:
sold_flagged["close_to_original_list_ratio"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

count    4.469430e+05
mean     5.094921e+01
std      1.476845e+04
min      9.207366e-07
1%       7.476240e-01
5%       8.598131e-01
25%      9.537526e-01
50%      9.956622e-01
75%      1.019355e+00
95%      1.115641e+00
99%      1.284596e+00
max      9.118000e+06
Name: close_to_original_list_ratio, dtype: float64

#### Add invalid_close_to_original_ratio_flag

In [68]:
# Add invalid close-to-original-list ratio flag
sold_flagged["invalid_close_to_original_ratio_flag"] = (
    (sold_flagged["close_to_original_list_ratio"] < 0.5)
    | (sold_flagged["close_to_original_list_ratio"] > 2)
)

In [69]:
sold_filtered = sold_flagged[
    ~sold_flagged["any_iqr_outlier_flag"]
].copy()

In [48]:
sold_filtered[
    [
        "ClosePrice",
        "OriginalListPrice",
        "close_to_original_list_ratio"
    ]
].sort_values(
    "close_to_original_list_ratio",
    ascending=False
).head(20)

,ClosePrice,OriginalListPrice,close_to_original_list_ratio
62274,1670000.0,1.55,1.077419e+06
166267,1149000.0,1.12,1.025893e+06
160167,1250000.0,1.25,1.000000e+06
240938,1255000.0,1.30,9.653846e+05
54465,1150000.0,1.20,9.583333e+05
130322,1220000.0,1.29,9.457364e+05
35253,983000.0,1.10,8.936364e+05
271664,600000.0,1.00,6.000000e+05
229060,1000000.0,10.00,1.000000e+05
72173,380000.0,4.00,9.500000e+04


In [61]:
sold_filtered[
    [
        "ListingId",
        "CloseDate",
        "ClosePrice",
        "OriginalListPrice",
        "close_to_original_list_ratio",
        "yrmo"
    ]
].sort_values(
    "close_to_original_list_ratio",
    ascending=False
).head(10)

,ListingId,CloseDate,ClosePrice,OriginalListPrice,close_to_original_list_ratio,yrmo
62274,TR24073775,2024-05-16,1670000.0,1.55,1.077419e+06,2024-05-01
166267,TR24187691,2024-11-27,1149000.0,1.12,1.025893e+06,2024-11-01
160167,PW24212946,2024-11-07,1250000.0,1.25,1.000000e+06,2024-11-01
240938,IG25078998,2025-05-30,1255000.0,1.30,9.653846e+05,2025-05-01
54465,TR24008917,2024-04-18,1150000.0,1.20,9.583333e+05,2024-04-01
130322,AR24151095,2024-09-20,1220000.0,1.29,9.457364e+05,2024-09-01
35253,DW24006419,2024-03-27,983000.0,1.10,8.936364e+05,2024-03-01
271664,OC25139061,2025-07-24,600000.0,1.00,6.000000e+05,2025-07-01
229060,RS25049210,2025-04-04,1000000.0,10.00,1.000000e+05,2025-04-01
72173,OC24039464,2024-05-10,380000.0,4.00,9.500000e+04,2024-05-01


In [62]:
monthly_ratio_check = (
    sold_filtered
    .groupby("yrmo")
    .agg(
        count=("close_to_original_list_ratio", "count"),
        mean_ratio=("close_to_original_list_ratio", "mean"),
        median_ratio=("close_to_original_list_ratio", "median"),
        max_ratio=("close_to_original_list_ratio", "max")
    )
    .reset_index()
)

monthly_ratio_check.sort_values(
    "mean_ratio",
    ascending=False
).head(10)

,yrmo,count,mean_ratio,median_ratio,max_ratio
10,2024-11-01,12083,169.066864,1.000000,1.025893e+06
8,2024-09-01,12686,77.265016,1.000000,9.457364e+05
4,2024-05-01,15774,75.609716,1.005112,1.077419e+06
16,2025-05-01,13203,74.649732,1.000000,9.653846e+05
2,2024-03-01,13373,68.545772,1.000910,8.936364e+05
3,2024-04-01,14689,66.530908,1.003110,9.583333e+05
18,2025-07-01,13393,46.782953,0.990000,6.000000e+05
15,2025-04-01,13184,9.572237,1.000000,1.000000e+05
11,2024-12-01,12092,9.082306,0.994092,9.416667e+04
24,2026-01-01,8017,2.401857,0.987828,8.260504e+03


In [53]:
print(
    "Ratio > 1.5:",
    (sold_filtered["close_to_original_list_ratio"] > 1.5).sum()
)

print(
    "Ratio > 2:",
    (sold_filtered["close_to_original_list_ratio"] > 2).sum()
)

print(
    "Ratio > 5:",
    (sold_filtered["close_to_original_list_ratio"] > 5).sum()
)

print(
    "Ratio > 10:",
    (sold_filtered["close_to_original_list_ratio"] > 10).sum()
)

print(
    "Ratio > 100:",
    (sold_filtered["close_to_original_list_ratio"] > 100).sum()
)

Ratio > 1.5: 1134
Ratio > 2: 534
Ratio > 5: 480
Ratio > 10: 280
Ratio > 100: 166


In [63]:
sold_filtered[
    sold_filtered["OriginalListPrice"] < 10000
][
    [
        "ListingId",
        "CloseDate",
        "ClosePrice",
        "OriginalListPrice",
        "close_to_original_list_ratio"
    ]
].sort_values(
    "OriginalListPrice"
).head(10)

,ListingId,CloseDate,ClosePrice,OriginalListPrice,close_to_original_list_ratio
356971,IV25173913,2025-12-29,267900.0,0.00,NaN
271664,OC25139061,2025-07-24,600000.0,1.00,6.000000e+05
35253,DW24006419,2024-03-27,983000.0,1.10,8.936364e+05
166267,TR24187691,2024-11-27,1149000.0,1.12,1.025893e+06
54465,TR24008917,2024-04-18,1150000.0,1.20,9.583333e+05
160167,PW24212946,2024-11-07,1250000.0,1.25,1.000000e+06
130322,AR24151095,2024-09-20,1220000.0,1.29,9.457364e+05
240938,IG25078998,2025-05-30,1255000.0,1.30,9.653846e+05
62274,TR24073775,2024-05-16,1670000.0,1.55,1.077419e+06
72173,OC24039464,2024-05-10,380000.0,4.00,9.500000e+04


In [71]:
print(
    sold_filtered["invalid_close_to_original_ratio_flag"].value_counts()
)

print(
    sold_filtered["invalid_close_to_original_ratio_flag"].value_counts(normalize=True) * 100
)

invalid_close_to_original_ratio_flag
False    376476
True        997
Name: count, dtype: int64
invalid_close_to_original_ratio_flag
False    99.735875
True      0.264125
Name: proportion, dtype: float64


In [72]:
sold_tableau = sold_filtered[
    ~sold_filtered["invalid_close_to_original_ratio_flag"]
].copy()

In [73]:
sold_tableau["close_to_original_list_ratio"].describe()

count    375905.000000
mean          0.997158
std           0.077609
min           0.500000
25%           0.963870
50%           1.000000
75%           1.022390
max           2.000000
Name: close_to_original_list_ratio, dtype: float64

In [74]:
(
    sold_tableau
    .groupby("yrmo")["close_to_original_list_ratio"]
    .mean()
)

yrmo
2024-01-01    0.990838
2024-02-01    1.005699
2024-03-01    1.013807
2024-04-01    1.018078
2024-05-01    1.017216
2024-06-01    1.011105
2024-07-01    1.003714
2024-08-01    0.998175
2024-09-01    0.995807
2024-10-01    0.996893
2024-11-01    0.994550
2024-12-01    0.990910
2025-01-01    0.988771
2025-02-01    1.001181
2025-03-01    1.001996
2025-04-01    1.000250
2025-05-01    0.995650
2025-06-01    0.991842
2025-07-01    0.984588
2025-08-01    0.982740
2025-09-01    0.983747
2025-10-01    0.985454
2025-11-01    0.985887
2025-12-01    0.983871
2026-01-01    0.982178
2026-02-01    0.994221
2026-03-01    1.000420
2026-04-01    0.998912
2026-05-01    0.997918
2026-06-01    0.995633
Name: close_to_original_list_ratio, dtype: float64

### Save Full Flagged and Filtered Datasets

In [75]:
# Define output file paths

SOLD_FLAGGED_OUTPUT = (
    OUTPUT_DIR
    / "sold_full_flagged.csv"
)

SOLD_FILTERED_OUTPUT = (
    OUTPUT_DIR
    / "sold_filtered.csv"
)

LIST_FLAGGED_OUTPUT = (
    OUTPUT_DIR
    / "list_full_flagged.csv"
)

LIST_FILTERED_OUTPUT = (
    OUTPUT_DIR
    / "list_filtered.csv"
)

In [76]:
# Save the full Sold dataset with IQR outlier flags
sold_flagged.to_csv(
    SOLD_FLAGGED_OUTPUT,
    index=False
)

# Save the Sold analysis dataset without IQR outliers
sold_filtered.to_csv(
    SOLD_FILTERED_OUTPUT,
    index=False
)

# Save the full List dataset with IQR outlier flags
list_flagged.to_csv(
    LIST_FLAGGED_OUTPUT,
    index=False
)

# Save the List analysis dataset without IQR outliers
list_filtered.to_csv(
    LIST_FILTERED_OUTPUT,
    index=False
)

all_files_saved = all([
    SOLD_FLAGGED_OUTPUT.exists(),
    SOLD_FILTERED_OUTPUT.exists(),
    LIST_FLAGGED_OUTPUT.exists(),
    LIST_FILTERED_OUTPUT.exists()
])

print(
    "All datasets saved successfully:",
    all_files_saved
)

All datasets saved successfully: True
